## 0 · One Kick, One Question

> **Match day. 20 metres from goal. One free kick.**
>
> The ball draws one smooth curve, but a simulation does not need to know the whole curve in advance. It can build the flight from many tiny questions:
>
> 1. Where is the ball now?
> 2. How far does horizontal velocity carry it during a tiny time slice?
> 3. How far does vertical velocity carry it during that same slice?
> 4. How much does gravity change vertical velocity for the next slice?
> 5. Repeat.
>
> That is calculus in action: use change **at this instant** to predict the **next small step**, then let many small steps accumulate into a journey.

The animation uses 28 equal time slices. A cyan bar computes the next horizontal change, a gold bar computes the next vertical change, and gravity prepares the vertical velocity used by the following step.

**Do not study the numbers yet. Watch the construction:** the vertical bars begin positive, shrink near the top, pass through zero, and become negative as the ball falls.

![Free-kick trajectory assembled from tiny horizontal and vertical change bars, with the next point calculated at every step](images/free-kick-forward-telemetry.gif)

**Watch the colors, not the numbers:**

- the cyan bar asks, “How far forward during this tiny moment?”
- the gold bar asks, “Will the next point be higher or lower?”
- the endpoint becomes the next starting point,
- gravity makes each later gold bar a little less upward, then downward.

No single bar knows the whole curve. The journey appears because the same local question is asked again and again.

<small>Match photograph: Michael Barera, [“Detroit City FC v. San Antonio FC 2023 20 (free kick)”](https://commons.wikimedia.org/wiki/File:Detroit_City_FC_v._San_Antonio_FC_2023_20_(free_kick).jpg), Wikimedia Commons, [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). Cropped, color-graded, and composited with physics-driven motion and instructional overlays.</small>

# Mathematical Foundations for Machine Learning

## Follow the Match in Time

The mathematics will arrive in the same order as the kick.

### 1. Before contact: choose a direction

The player sees the ball, wall, goal, and target. An **arrow** is the natural way to record a direction and a strength. Mathematics calls that arrow a **vector**.

Two arrows now matter:

- where the kick points;
- where the intended route points.

We need a score for how much one arrow supports the other. That score will become the **dot product**, but the picture comes first.

### 2. After contact: predict the next instant

Once the ball is moving, its current velocity predicts one nearby point. This is the local-change idea behind a **derivative**.

### 3. During the flight: repeat

One nearby prediction is not a journey. Repeating and adding the tiny changes builds the full arc. This is the accumulation idea behind **integration**.

### 4. Across many kicks: allow variation

One exact kick follows one curve. Repeated kicks form a spread of possible outcomes, which introduces **probability**.

### 5. At Challenger Deep: improve a control

The probe uses a local derivative to decide how a tiny thrust change affects mission score. Repeated downhill parameter steps become **gradient descent**.

> **First see the football question. Then give its mathematical tool a name.**

In [ ]:
# Dependencies
import subprocess, sys

# Only install packages that are not already importable in this environment
required = [("numpy", "numpy"), ("scipy", "scipy")]
for imp, pkg in required:
    try:
        __import__(imp)
        print(f"  ok  {pkg}")
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        print(f"  done {pkg}")

import numpy as np
from scipy import stats

np.random.seed(42)
print("All dependencies ready")

# Physical constants for the free kick scenario
g = 9.81          # gravity (m/s²)
v0 = 20.0        # launch speed (m/s)
BALL_RADIUS = 0.11
WALL_X = 9.15    # wall position (m)
WALL_H = 1.8     # wall height (m)
GOAL_X = 20.0    # front of the goal (m)
CROSS_H = 2.44   # crossbar height (m)
NET_X = 21.5     # back of the net (m)
TARGET_H = 1.10  # intended ball-centre height at back-net impact (m)
GOAL_BOTTOM = BALL_RADIUS
GOAL_TOP = CROSS_H - BALL_RADIUS


def ball_state(x, theta_deg):
    """Return time, height, and velocity when the ball reaches horizontal position x."""
    theta = np.radians(theta_deg)
    vx = v0 * np.cos(theta)
    t = x / vx
    vy = v0 * np.sin(theta) - g * t
    y = v0 * np.sin(theta) * t - 0.5 * g * t**2
    return {"x": float(x), "t": float(t), "y": float(y), "vx": float(vx), "vy": float(vy)}


def ball_height(x, theta_deg):
    """Height of the ball centre at horizontal position x and launch angle theta (degrees)."""
    return ball_state(x, theta_deg)["y"]


print("\nFree kick setup:")
print(f"  Launch speed: {v0} m/s")
print(f"  Wall at {WALL_X}m, ball centre must clear {WALL_H + BALL_RADIUS:.2f}m")
print(f"  Goal plane at {GOAL_X}m, legal centre window [{GOAL_BOTTOM:.2f}, {GOAL_TOP:.2f}]m")
print(f"  Back-net target at ({NET_X:.1f}m, {TARGET_H:.2f}m)")

---

## Part 1 — Before the Kick: Compare Two Arrows

The player has not kicked yet. First, choose a direction.

A single angle such as 25° is compact, but an arrow is easier to reason with:

- the arrow's **direction** says where the kick points;
- the arrow's **length** can represent strength;
- its horizontal and vertical parts tell us how much motion goes each way.

An arrow used this way is a **vector**.

Now place two unit-length vectors at the ball:

1. the kick vector;
2. the intended-route vector.

The player needs more than “the angles look close.” We want one signed score:

- strongly positive when the kick points along the route;
- zero when it points sideways;
- negative when it points backward.

![Dot product visualized as the signed shadow of the kick arrow along the intended route](images/free-kick-vector-projection.svg)

The green segment is the kick arrow's forward shadow on the target line. That shadow is the intuition behind the **dot product**.

#### Predict First

The kick points at 25°. The intended route points at 20°. Their difference is only 5°.

What should the shadow look like?

1. Almost as long as the full arrow
2. About half the arrow
3. Pointing backward

Make the visual prediction before opening the notation.

### Let the Shadow Become a Number

The kick and route differ by only **5°**. In the diagram, the green shadow is therefore almost as long as the kick arrow.

Suppose every arrow is scaled to length `1`:

- a full forward shadow would score `1`;
- this nearly full shadow scores about `0.996`;
- no forward shadow would score `0`;
- a full backward shadow would score `−1`.

The familiar geometry function that converts an angle gap into this shadow fraction is **cosine**:

- `cos(0°) = 1` — full agreement;
- `cos(90°) = 0` — sideways;
- `cos(180°) = −1` — opposite;
- `cos(5°) ≈ 0.996` — our kick.

Only now compress the idea:

$$\text{shadow score}=\cos(\text{angle between the arrows})$$

The mathematical name for the shadow score is the **dot product**. Using short vector names gives:

$$\mathbf{kick}\cdot\mathbf{route}=\cos(\text{angle between them})$$

The notation records the picture; it does not replace it.

<details>
<summary><strong>How the computer recovers the same shadow from coordinates</strong></summary>

The 25° kick arrow is stored approximately as:

> horizontal part `0.906`, vertical part `0.423`

The 20° route arrow is stored approximately as:

> horizontal part `0.940`, vertical part `0.342`

Compare matching directions and add their agreements:

$$0.906\times0.940+0.423\times0.342\approx0.996$$

That is the same shadow score predicted by the diagram.

Now name the storage pattern. A 2D vector keeps its two components together:

$$\mathbf{v}=[v_x,v_y]$$

For any two vectors, multiply matching components and add:

$$\mathbf{a}\cdot\mathbf{b}=a_xb_x+a_yb_y$$

The general formula grows directly from the numerical kick example.

</details>

### Read the Score Like a Player

| Arrow relationship | Shadow on the intended route | Meaning |
| --- | --- | --- |
| Same direction | Long and forward | Most of the kick helps the intended route. |
| Nearly aligned | Almost full length | A small angle difference loses little forward agreement. |
| Sideways | Almost no length | The kick does not help along the intended route. |
| Opposite | Points backward | The kick fights the intended route. |

For the 25° kick and 20° route, expect a score very close to `+1`.

Only now do we need the code. It converts each angle into a two-component vector and calculates the shadow score numerically.

In [ ]:
# Part 1: turn two angles into unit vectors and measure their alignment
kick_dir = np.array([np.cos(np.radians(25)), np.sin(np.radians(25))])
route_dir = np.array([np.cos(np.radians(20)), np.sin(np.radians(20))])

# For unit vectors, the dot product is the signed projection length.
shadow_score = np.dot(kick_dir, route_dir)
angle_between = np.degrees(np.arccos(np.clip(shadow_score, -1, 1)))

print(f"Kick vector:  {kick_dir.round(3)}")
print(f"Route vector: {route_dir.round(3)}")
print(f"Angle between arrows: {angle_between:.1f}°")
print(f"Forward-shadow score: {shadow_score:.4f} out of 1")
print()
print("Interpretation: almost the entire kick arrow points along the intended route.")

#### What the Number Confirms

The score is **0.9962**, very close to the maximum value of `1`. The picture predicted this: the kick arrow casts an almost full-length forward shadow onto the intended route.

The dot product has completed one job:

> **Compare two directions at the instant of the kick.**

It does not predict the ball's later position. Once the ball leaves the foot, the question changes from alignment to motion:

> **Given the velocity now, where is the next nearby point?**

That question leads to derivatives and the close-up staircase animation in Part 2.

<details>
<summary><strong>Try a different direction</strong></summary>

```python
# Try 90° for a mostly sideways kick, then 200° for a backward-pointing kick.
kick_angle = 90
goal_angle = 20

kick = np.array([np.cos(np.radians(kick_angle)), np.sin(np.radians(kick_angle))])
goal = np.array([np.cos(np.radians(goal_angle)), np.sin(np.radians(goal_angle))])
score = np.dot(kick, goal)
print(f"kick={kick_angle}°, route={goal_angle}° → shadow score={score:.4f}")
```

</details>

---

## Part 2 — After Contact: Build One Nearby Move

The ball has left the foot. Directional agreement is no longer the question. Now ask:

> **Given the ball's motion at this point, where should the next nearby point be?**

![Close-up animation following the ball while a small cyan-and-gold rectangular step is constructed at every point](images/free-kick-step-closeup.gif)

At every green **NOW** point, the same calculation is drawn:

1. the **cyan edge** records the forward change during one tiny time slice;
2. the **gold edge** records the height change during that same slice;
3. the opposite corner becomes **NEXT**;
4. NEXT becomes the next NOW, and the calculation repeats.

The rectangle is a measurement scaffold. The ball does not physically travel sideways and then vertically; the two perpendicular edges split one nearby displacement into horizontal and vertical components.

Watch the gold edges:

- early rectangles rise strongly;
- later rectangles rise less;
- near the top, the gold edge almost vanishes;
- after the top, it points downward.

That changing local move is the intuition behind a **derivative**.

#### Predict First

For the 25° shot, where will the gold edge change from upward to downward?

1. Around the wall at 10 m
2. Between the wall and goal at 15–16 m
3. At the goal line near 20 m

Make the visual prediction, then run the code cell.

In [ ]:
# Part 2: build the trajectory from the same tiny updates shown in the animation
theta_deg = 25.0
theta_rad = np.radians(theta_deg)
step_count = 28

velocity_x = v0 * np.cos(theta_rad)
position_x = 0.0
position_y = 0.0
velocity_y = v0 * np.sin(theta_rad)
flight_time = NET_X / velocity_x
delta_t = flight_time / step_count

step_rows = []
for step_index in range(step_count):
    delta_x = velocity_x * delta_t
    delta_y = velocity_y * delta_t - 0.5 * g * delta_t**2
    next_velocity_y = velocity_y - g * delta_t

    if step_index in {0, 9, 18, 27}:
        step_rows.append((step_index + 1, position_x, position_y, delta_x, delta_y, next_velocity_y))

    position_x += delta_x
    position_y += delta_y
    velocity_y = next_velocity_y

# Compare the accumulated small steps with the direct projectile formula
net_state = ball_state(NET_X, theta_deg)
step_error = np.hypot(position_x - net_state["x"], position_y - net_state["y"])

print(f"28 slices across {flight_time:.3f}s give Δt={delta_t:.3f}s\n")
print(" step   start (x, y)       Δx        Δy       next vy")
for step_number, start_x, start_y, change_x, change_y, next_vy in step_rows:
    print(
        f" {step_number:>2d}    ({start_x:>5.2f}, {start_y:>4.2f})   "
        f"{change_x:+.3f}m   {change_y:+.3f}m   {next_vy:+.3f}m/s"
    )

print(f"\nAccumulated endpoint: ({position_x:.6f}, {position_y:.6f}) m")
print(f"Direct-formula endpoint: ({net_state['x']:.6f}, {net_state['y']:.6f}) m")
print(f"Endpoint disagreement: {step_error:.2e} m")
assert step_error < 1e-10

x_peak_analytic = v0**2 * np.sin(2 * theta_rad) / (2 * g)
peak_state = ball_state(x_peak_analytic, theta_deg)
print(f"\nTurning point: x={x_peak_analytic:.2f}m, y={peak_state['y']:.2f}m, vy={peak_state['vy']:+.2e}m/s")
print("Prediction check: answer 2 — Δy changes sign between the wall and goal.")

### Repeat the Same Nearby Question

One rectangle gives one next point. To build a journey, make that endpoint the new **NOW** and ask again.

![Animation showing 4, 8, 16, and 28 repeated local predictions becoming a progressively smoother free-kick curve](images/calculus-accumulation.gif)

Read the stages as a progression:

- **4 large steps:** the rough direction is visible, but the path is angular.
- **8 steps:** more nearby guesses follow the bend.
- **16 steps:** the staircase hugs the curve.
- **28 tiny steps:** many local changes assemble the full flight.

Before introducing symbols, inspect one real rectangle from the code:

- the time slice lasts about `0.042 s`;
- horizontal speed carries the ball about `0.768 m` forward;
- near step 10, vertical motion adds about `0.191 m` of height.

So the nearby move is simply:

> **next position = current position + forward change + height change**

Now name the pieces:

- $\Delta t$ means the tiny time slice;
- $v_x$ and $v_y$ mean the current horizontal and vertical velocities;
- $\Delta x$ and $\Delta y$ mean the resulting position changes.

For one sufficiently small slice:

$$\Delta x\approx v_x\Delta t,\qquad \Delta y\approx v_y\Delta t$$

Velocity itself means position change per unit time:

$$v_x=\frac{dx}{dt},\qquad v_y=\frac{dy}{dt}$$

The derivative names the local rate. Repeatedly adding the small changes is **integration**.

> **Derivative: read one local move. Integration: accumulate all local moves.**

#### Two Calculus Moves

The animations separate one continuous idea into two jobs:

1. **Read the present:** use the ball's current motion to predict one nearby point.
2. **Build the journey:** make that point the new starting point and repeat.

No single step contains a blueprint of the full arc. It only needs the ball's current state, gravity, and one short time slice.

Real football flight also depends on drag, spin, lift, and wind. Those forces would change each prediction, but not the pattern:

> **read the current state → predict the next state → update → repeat**

A useful next question is therefore: how do we carry several pieces of state—position, velocity, time, and acceleration—together? Vectors collect them; matrices transform them.

---

## Part 3 — One Recipe, Then Many Recipes

The flight calculation carries several related values together. For a smaller example, keep only:

> angle signal `0.4`, speed signal `0.8`

Putting them in a fixed order creates one input vector:

> **input vector = [angle signal, speed signal] = [0.4, 0.8]**

Now suppose we want two internal answers:

1. a clearance signal;
2. a target-height signal.

One weighted recipe can produce one answer:

> multiply each input by its relevance, add the pieces, then add an offset

That is exactly the dot-product move from Part 1, now used as a prediction recipe rather than an alignment score.

To produce both answers, stack two recipes:

![Matrix multiplication built from two stacked dot-product recipes applied to an angle-and-speed state vector](images/free-kick-matrix-recipes.svg)

This stack is called a **matrix**:

- one row stores one recipe;
- each column follows one input through all recipes;
- all row answers form the output vector.

#### Predict First

If both recipes use the angle input, what happens when only angle changes?

1. Only the first output can change
2. Every output connected to angle can change
3. Neither output changes

Use the diagram and the row calculations below before opening the compact notation.

> **Connection to Part 1:** a dot product runs one row. Matrix multiplication runs every row against the same input vector.

> **Connection to the kick:** angle and speed enter together. Different rows can listen to them differently, so one shared state can produce several useful signals.

In [ ]:
# Part 3: stack two dot-product recipes into one matrix
features = np.array([0.4, 0.8])  # normalized [angle signal, speed signal]

# Each row is one dot-product recipe applied to the same input vector.
W = np.array([
    [2.0, 0.5],   # row 1 listens strongly to angle
    [-0.5, 1.5],  # row 2 listens strongly to speed
])
b = np.array([0.1, -0.2])

output = W @ features + b

print(f"Input vector [angle, speed]: {features}")
print("\nRun each row as one dot product:")
for row_index, (row, bias, result) in enumerate(zip(W, b, output), start=1):
    angle_part, speed_part = row * features
    print(
        f"  row {row_index}: angle {angle_part:+.2f}, speed {speed_part:+.2f}, "
        f"bias {bias:+.2f} → result {result:.2f}"
    )

print(f"\nStacked matrix result: {output.round(3)}")
print("One row gives one output; two rows give an output vector of length two.")

### Read the Rows Before the Matrix

The input vector is:

- angle signal = `0.4`;
- speed signal = `0.8`.

Each row asks its own weighted question:

| Recipe | Angle contribution | Speed contribution | Bias | Result |
| --- | ---: | ---: | ---: | ---: |
| Row 1 | `2 × 0.4 = 0.8` | `0.5 × 0.8 = 0.4` | `+0.1` | **1.3** |
| Row 2 | `−0.5 × 0.4 = −0.2` | `1.5 × 0.8 = 1.2` | `−0.2` | **0.8** |

Read one row at a time:

- Row 1 listens strongly to angle and a little to speed.
- Row 2 treats angle as a small brake and listens strongly to speed.
- The bias shifts the final answer after the dot product.

These are toy internal signals, not metres or scoring probabilities. Their job is to reveal the structure.

Now read by columns:

- the first column contains every route taken by angle;
- the second column contains every route taken by speed.

> **Rows build outputs. Columns trace inputs. The matrix stores both views at once.**

#### Let the Rows Become Matrix Notation

The code has already run the concrete recipes:

- Row 1 turns `[0.4, 0.8]` into `1.3`.
- Row 2 turns `[0.4, 0.8]` into `0.8`.
- Keeping both answers in order gives `[1.3, 0.8]`.

The conceptual ladder is:

1. **Number:** one measured value.
2. **Vector:** several related values kept in order.
3. **Dot product:** one weighted recipe applied to a vector.
4. **Matrix:** several weighted recipes stacked together.

Now assign compact names:

- $\mathbf{x}$ = input vector `[0.4, 0.8]`;
- $W$ = stack of row recipes;
- $\mathbf{b}$ = one offset for each row;
- $\mathbf{y}$ = output vector `[1.3, 0.8]`.

The whole calculation becomes:

$$\mathbf{y}=W\mathbf{x}+\mathbf{b}$$

Read it as a sentence, not a spell:

> **run every row of $W$ against the same input $\mathbf{x}$, then add each row's offset**

Prediction (2) is correct: because angle appears in both rows, changing angle can change both outputs.

<details>
<summary><strong>Try changing one column</strong></summary>

```python
W_test = np.array([
    [3.0, 0.1],  # row 1 listens mostly to angle
    [0.1, 3.0],  # row 2 listens mostly to speed
])

normal = np.array([0.4, 0.8])
more_angle = np.array([0.8, 0.8])

print("normal:    ", W_test @ normal)
print("more angle:", W_test @ more_angle)
print("change:    ", W_test @ more_angle - W_test @ normal)
```

The first output changes much more because its row listens strongly to the angle column.

</details>

So far each kick used exact inputs. Real players vary from attempt to attempt. That leads to probability.

---

## Part 4 — One Kick Is Not a Cloud of Attempts

So far the same angle and speed always produce the same curve. That describes a perfectly repeatable kick.

A real player is not perfectly repeatable. Ask for 20° one hundred times and the actual kicks form a cloud around 20°:

- many land close to the intention;
- some are a little high or low;
- a few are farther away.

That cloud is a **probability distribution**.

The legal scoring angles span roughly **18.44° to 21.86°**. Compare two aims:

- **19.13°** places one exact kick near the chosen point;
- **20.15°** centres the whole cloud inside the legal window.

The visual question comes first:

> **Which centre leaves more of the cloud between the two legal boundaries?**

#### Predict First

Suppose the typical miss is about 3°. Which aim scores more often over 100 attempts?

1. 19.13°, because it is best for one exact kick
2. 20.15°, because it centres the whole cloud inside the legal window
3. They must be identical

<details>
<summary><strong>Let the cloud become probability notation</strong></summary>

Give the uncertain executed angle a name: $\Theta$.

The statement

> executed angle lands between 18.44° and 21.86°

becomes

$$18.44°<\Theta<21.86°$$

Placing $P(\cdot)$ around an event means “the probability of this event”:

$$P(18.44°<\Theta<21.86°)$$

So this expression means exactly:

> **the fraction of the angle cloud that falls inside the legal window**

For the centred 3° cloud, that fraction is about `0.431`, or 43 scores per 100 attempts.

Now turn probability into a surprise score. A likely event should have a small penalty; an unlikely event should have a large penalty. Negative logarithm has that behavior:

- chance `0.9` → small surprise;
- chance `0.1` → large surprise.

Name the chance $p$. Then:

$$\text{surprise}=-\log(p)$$

When $p$ is the probability assigned to the observed outcome, this is **negative log-likelihood**.

</details>

> **Keep the objects separate:** one exact input produces one trajectory. Repeated imperfect inputs produce a cloud of trajectories. Probability describes the cloud rather than pretending every attempt is identical.

In [ ]:
# Part 4: compare one point-target aim with a reliable repeated-attempt aim
# Keep angles that clear the wall and cross fully inside the goal frame
scoreable_angles = [
    angle for angle in np.linspace(5, 60, 20_000)
    if (
        ball_height(WALL_X, angle) > WALL_H + BALL_RADIUS
        and GOAL_BOTTOM < ball_height(GOAL_X, angle) < GOAL_TOP
    )
]

if not scoreable_angles:
    raise RuntimeError("No launch angle satisfies the physical constraints")

theta_lo = min(scoreable_angles)
theta_hi = max(scoreable_angles)
point_target_aim = 19.13
cluster_centered_aim = (theta_lo + theta_hi) / 2
sigma = 3.0


def probability_of_scoring(intended_angle, execution_sigma=sigma):
    distribution = stats.norm(loc=intended_angle, scale=execution_sigma)
    return distribution.cdf(theta_hi) - distribution.cdf(theta_lo)


probability_point_target = probability_of_scoring(point_target_aim)
probability_cluster_centered = probability_of_scoring(cluster_centered_aim)
nll_point_target = -np.log(probability_point_target)
nll_cluster_centered = -np.log(probability_cluster_centered)

print(f"Legal angle window: [{theta_lo:.2f}°, {theta_hi:.2f}°]")
print(f"Typical execution spread: {sigma:.1f}°\n")
print("ONE EXACT INPUT VERSUS A DISTRIBUTION")
print(
    f"  Aim at one chosen point: {point_target_aim:5.2f}°  "
    f"expected scores per 100={100 * probability_point_target:4.1f}"
)
print(
    f"  Center the whole cluster: {cluster_centered_aim:5.2f}°  "
    f"expected scores per 100={100 * probability_cluster_centered:4.1f}"
)
print(f"  Extra expected scores per 100: {100 * (probability_cluster_centered - probability_point_target):+.1f}")
print(f"  Surprise penalty: {nll_point_target:.3f} → {nll_cluster_centered:.3f}")
print("\nPrediction check: answer (b) — centering the cluster leaves more room for imperfect kicks.\n")

print("Consistency check at the cluster-centered aim:")
for spread in [1.0, 2.0, 3.0, 5.0, 8.0, 10.0]:
    probability = probability_of_scoring(cluster_centered_aim, spread)
    print(f"  typical spread={spread:4.1f}°: expected scores per 100={100 * probability:4.1f}")

### Imagine 100 repeated kicks

With a 3° typical spread, the two aims behave differently over many attempts:

| Strategy | Intended angle | Expected scores out of 100 | What it describes |
| --- | ---: | ---: | --- |
| Hit one back-net point precisely | 19.13° | About **41** | One exact-trajectory target |
| Center varied attempts in the legal window | 20.15° | About **43** | Probability of any legal score |

The gain is only about two extra goals per 100 kicks here, but the principle is durable:

> **A single predicted value and a distribution of possible values answer different questions.**

Consistency matters even more than the two-degree change in aim:

| Typical spread around the aim | Rough scores out of 100 at the centered aim | What the cluster looks like |
| ---: | ---: | --- |
| 1° | **91** | Tight and consistent |
| 2° | **61** | Wider, but most attempts remain near the aim |
| 3° | **43** | Many attempts spill outside the narrow legal window |
| 5° | **27** | Broad scatter |
| 8° | **17** | Very broad scatter |

Moving the **aim** shifts the centre of the distribution. Improving **consistency** tightens it. Probability lets us reason about both without pretending every repeated attempt is identical.

#### What just happened

The two questions preferred different reference angles:

- **19.13°** places one exact kick near the chosen back-net point,
- **20.15°** gives a spread of imperfect kicks the most room inside the legal interval.

Over 100 attempts with a 3° typical spread, that shift raises expected scores from about **41 to 43**. The numerical surprise penalty also falls from about **0.892 to 0.841**; smaller means a legal score is less surprising under the distribution.

This is the bridge to machine learning:

> **A deterministic prediction says what one input produces. A probabilistic model says how plausible many possible outcomes are.**

#### Your turn — cost of poor consistency

Before changing the number, predict what a wider cluster will do:

```python
typical_spread = 8.0  # change to 1.0 after predicting the effect
chance = probability_of_scoring(cluster_centered_aim, typical_spread)
print(f"typical spread={typical_spread}°: expected scores per 100={100 * chance:.1f}")
print(f"surprise penalty={-np.log(chance + 1e-12):.3f}")
```

This calculation does not claim every football error follows a perfect bell curve. It shows how an assumption about uncertainty becomes a testable probability.

<details>
<summary><strong>Connect the surprise penalty to classification</strong></summary>

The formal name for the surprise penalty is **negative log-likelihood**. In classification, cross-entropy is the negative log probability assigned to the correct class.

The Gaussian model here and the softmax probabilities used for classification are different distributions, but both use the same principle: **assign high probability to outcomes like the ones observed**.

</details>

---

## Part 5 — From a Local Derivative to Gradient Descent

The football derivative answered:

> “Given the motion now, where is the next nearby point?”

Challenger Deep adds an adjustable thrust control. Its derivative answers:

> “If thrust changed a tiny amount here, would the mission plan become better or worse?”

The probe must reach roughly **10,900 metres** while balancing:

- **distance:** close the remaining gap;
- **time:** avoid exhausting the underwater window;
- **pressure:** avoid unnecessarily aggressive thrust in deep water.

The mission computer needs one ruler for comparing plans. It adds three penalties:

> **mission score = distance penalty + time penalty + pressure penalty**

Lower is better. A plan can improve one concern while worsening another, but the total score lets us compare the trade-off.

Now give the two changing quantities short names:

- $u$ means the thrust control;
- $L$ means the resulting mission score.

Writing $L(u)$ means:

> **the mission score produced when the control is $u$**

The verbal ruler can now be compressed:

$$L(u)=\text{distance penalty}+\text{time penalty}+\text{pressure penalty}$$

Gradient descent changes $u$. It does not change physical depth directly. After the parameter update is checked, the new control produces one physical probe step.

### Derivation 1 — Discover the Sign from Two Tiny Tests

First separate the two kinds of movement:

![Diagram separating a gradient step in parameter space from the probe's physical step through ocean depth](images/challenger-gradient-two-spaces.svg)

The physical probe has not moved. We are only testing the score near the current thrust setting.

![Animation deriving the local derivative sign by comparing tiny less-thrust and more-thrust tests at the surface and in deep water](images/challenger-derivative-sign-test.gif)

At the surface, the animation reports:

- current score: `0.9586`;
- score after a tiny move left: `0.9760`;
- score after a tiny move right: `0.9430`.

Moving right lowered the score. Therefore the score curve slopes downward from left to right, so the derivative is negative.

Near 8,600 m the comparison reverses:

- tiny move left: score `0.0466`;
- current score: `0.0477`;
- tiny move right: score `0.0487`.

Now moving left lowers the score, so the derivative is positive.

### Turn the Comparison into a Rate

The test used two controls separated by `0.24`: one `0.12` left of NOW and one `0.12` right.

At the surface:

$$\frac{\text{score change}}{\text{control change}}\approx\frac{0.9430-0.9760}{0.24}\approx-0.1375$$

The negative result matches the animation.

Now replace the concrete test distance `0.12` with the symbol $\varepsilon$:

- $u-\varepsilon$ means a tiny move left;
- $u+\varepsilon$ means a tiny move right;
- the full tested distance is $2\varepsilon$.

The nearby slope estimate becomes:

$$\frac{dL}{du}\approx\frac{L(u+\varepsilon)-L(u-\varepsilon)}{2\varepsilon}$$

As the test distance shrinks, this estimate approaches the derivative at NOW.

> **The derivative sign identifies uphill. Gradient descent chooses the opposite direction.**

### Derivation 2 — Direction Is Not Enough; Choose a Step Size

At 8,600 m, the local derivative is about `+0.0088`.

Positive means moving thrust right raises the score. The downhill direction is therefore left.

But **how far left?** The derivative is local advice, not permission to jump anywhere on the curve.

![Animation comparing a tiny, useful, and oversized step taken along the same downhill derivative direction](images/challenger-step-size-comparison.gif)

The animation temporarily removes the game's safety cap so the three outcomes are visible:

| Control move | New score | Interpretation |
| ---: | ---: | --- |
| `−0.088` | `0.0469` | Improves, but barely moves. |
| `−1.059` | `0.0415` | Lands near the local valley. |
| `−2.118` | `0.0547` | Crosses the valley and becomes worse. |

All three moves use the same downhill direction. Their difference is only scale.

Call the chosen scale the **learning rate**. Use the Greek letter $\eta$ (eta) as its short name.

The verbal rule is:

> **control change = − learning rate × local derivative**

Now substitute one row from the animation. With derivative `+0.0088` and scale `120`:

$$\text{control change}\approx-120\times0.0088\approx-1.06$$

Only after seeing the numerical move do we shorten the names:

- $\Delta u$ means control change;
- $dL/du$ means the local derivative;
- $\eta$ means learning rate.

Therefore:

$$\Delta u=-\eta\frac{dL}{du}$$

Finally, add the change to the current control:

$$u_{new}=u+\Delta u=u-\eta\frac{dL}{du}$$

The actual game caps $|\Delta u|$ at `0.18`, preventing the oversized jump shown in the comparison.

### Derivation 3 — Recalculate After Every Physical Step

The update does not finish the mission. It chooses the control for one short physical move.

![Close-up gradient-descent animation showing one local control-and-score rectangle before each physical probe move](images/challenger-gradient-step-closeup.gif)

Each turn repeats:

1. measure the local derivative;
2. choose the negative-gradient direction;
3. scale it into a bounded control step;
4. confirm that the predicted score falls;
5. move the probe for ten minutes;
6. rebuild the score curve at the new depth.

The close-up shows why remeasurement matters. Early derivatives are negative, so increasing thrust helps. Near the bottom they become positive, so the improving direction reverses and thrust decreases.

> **Gradient descent is not “keep moving the same way.” It is “measure here, move a little downhill, then measure again.”**

### Concept Check

| Question | Answer |
| --- | --- |
| What does the derivative provide? | The local uphill direction and steepness. |
| Why subtract it? | Subtraction points the update downhill. |
| What does the learning rate provide? | The step size. |
| Why cap the move? | Local slope information should not authorize a reckless jump. |
| Why recompute after the probe moves? | The mission score changes with depth, pressure, and remaining time. |

<details>
<summary><strong>Optional practice: pilot the probe yourself</strong></summary>

The notebook has already derived the concepts. The external game is a practice environment where you can follow or deliberately fight the derivative.

<a href="challenger-deep-gradient-game.html" target="_blank"><strong>Open the optional Challenger Deep gradient game</strong></a>

Suggested experiment:

1. Follow the negative derivative once and observe the score fall.
2. Reset and choose the opposite direction once; observe the score rise.
3. Reset and complete the 14-turn mission.

</details>

The football and probe now answer two related local-change questions:

- **Football:** how is position changing right now?
- **Probe:** how would mission score change if thrust moved slightly?

ML Basics comes next and applies repeated downhill updates to fitted models.

→ **Next:** [`../01-ml-basics/ml-basics.ipynb`](../01-ml-basics/ml-basics.ipynb) — use local gradients to fit regression and classification models.